# Phase 4: Comparative Study (Direct vs Generative | Standard vs Wavelet)

## Abstract
This notebook presents the formal evaluation of the **Grand Tournament**, comparing four architectures across two axes: **Backbone Capability** (Direct vs Generative) and **Decoder Inductive Bias** (Standard CNN vs Ricker Wavelet).

## 1. The Contestants
1. **Mason-CNN** (Direct + CNN): State-of-the-art Deterministic Baseline.
2. **cNVAE-ECG** (Generative + CNN): State-of-the-art Generative Baseline.
3. **Mason-Ricker** (Direct + NIWT): Hybrid Control (Isolates Decoder effect).
4. **cNVAE-Ricker** (Generative + NIWT): Proposed Innovation.

## 2. Evaluation Strategy
- **Fidelity**: MAE, MSE, Pearson Correlation.
- **Clinical Utility**: AUROC (Acute MI Detection) using `MasonClassifier` (8-lead specialist) and `XResNet101` (12-Lead Generalist).

In [ ]:
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Load ACTUAL Evaluation Results from Scientific Table
results_path = "../logs/final_scientific_table.json"

if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        raw = json.load(f)
    
    # Transform nested structure: {model: {phase: metrics}}
    data = []
    for model, phases in raw.items():
        if 'phase2' in phases:  # Show best phase
            entry = phases['phase2'].copy()
            entry['Model'] = model
            entry['test_mae'] = entry.get('mae', 0)
            entry['test_corr'] = entry.get('pearson', 0)
            data.append(entry)
    
    df = pd.DataFrame(data)
    print('=== ACTUAL Results from final_scientific_table.json ===')
    display(df[['Model', 'test_mae', 'test_corr', 'auroc']].sort_values('auroc', ascending=False))
else:
    print(f'Results file not found: {results_path}')
    df = pd.DataFrame()


## 3. Clinical Utility Analysis (AUROC)
The primary hypothesis is that Ricker Wavelets preserve diagnostic features better than point-wise CNNs.

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x='Model', y='auroc', palette='viridis')
plt.title("Clinical Utility (Acute MI Detection AUROC)")
plt.ylim(0.5, 1.0)
plt.axhline(0.92, color='r', linestyle='--', label='Original Signal (Oracle)')
plt.legend()
plt.show()

## 4. Signal Fidelity (MAE and Correlation)
While AUROC measures utility, Correlation measures shape preservation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(data=df, x='Model', y='test_mae', ax=ax[0], palette='Reds_r')
ax[0].set_title("Reconstruction Error (MAE) - Lower is Better")

sns.barplot(data=df, x='Model', y='test_corr', ax=ax[1], palette='Blues')
ax[1].set_title("Waveform Correlation (Pearson) - Higher is Better")

plt.show()